In [2]:
import sys
import re
from pathlib import Path
from collections import defaultdict
from bs4 import BeautifulSoup

# Add utils to path
sys.path.append(str(Path.cwd().parent / 'llm_based_annotation'))
from utils_extraction.html_utils import is_manual_label_tag
from utils_extraction.htmlLabel import HTMLLabel

In [3]:
# Define the three HTML files to analyze
from pathlib import Path
data_dir = Path.cwd().parent / 'data' / 'Documents_Annotés'
final_annotated_dir = Path.cwd().parent / 'data' / 'final' / 'Annotated'

html_files = [
    data_dir / 'EG' / '1997CanLII16226_ONCA_annotated_EG_tech.html',
    data_dir / 'EG' / '2021QCCA1675_annotated_EG_tech.html',
    data_dir / 'GL' / '1989CanLII1415CITT_annotated_GL.html',
    final_annotated_dir / '2001CanLII21117QCTDP_annotated_GL_tech.html',
    final_annotated_dir / "2019SCC65_annotated_EG_revRL_tech.html",
    final_annotated_dir / "2024NBKB203_annotated_VP.html"



]

# Verify files exist
for f in html_files:
    print(f"{'✓' if f.exists() else '✗'} {f.name}")

✓ 1997CanLII16226_ONCA_annotated_EG_tech.html
✓ 2021QCCA1675_annotated_EG_tech.html
✓ 1989CanLII1415CITT_annotated_GL.html
✓ 2001CanLII21117QCTDP_annotated_GL_tech.html
✓ 2019SCC65_annotated_EG_revRL_tech.html
✓ 2024NBKB203_annotated_VP.html


In [4]:
def extract_parent_level_annotations(html_content):
    """
    Extract all parent-level manual_label annotations (where parent="").
    Returns a dict with keys: 'decision', 'legislation', 'secondary sources'
    Each value is a list of annotation dictionaries containing:
      - full_html: the complete annotation HTML
      - docid: document identifier
      - uri: resource URI
      - text_content: extracted text (no HTML tags)
      - sublabels: list of sublabel types found
    """
    soup = BeautifulSoup(html_content, 'html.parser')
    
    # Find all manual_label tags with parent=""
    parent_labels = soup.find_all('manual_label', attrs={'parent': ''})
    
    annotations = {
        'decision': [],
        'legislation': [],
        'secondary sources': []
    }
    
    for label in parent_labels:
        labelname = label.get('labelname', '')
        
        if labelname not in annotations:
            continue
        
        # Extract sublabels
        sublabels = []
        for sublabel in label.find_all('manual_label', recursive=False):
            sublabel_name = sublabel.get('labelname', '')
            sublabels.append(sublabel_name)
        
        # Recursively get all sublabels (nested)
        all_sublabels = [sl.get('labelname', '') for sl in label.find_all('manual_label')]
        
        annotation_data = {
            'full_html': str(label),
            'docid': label.get('docid', ''),
            'uri': label.get('uri', ''),
            'text_content': label.get_text(strip=True),
            'direct_sublabels': sublabels,
            'all_sublabels': all_sublabels
        }
        
        annotations[labelname].append(annotation_data)
    
    return annotations

In [5]:
# Process all files
all_annotations = {
    'decision': [],
    'legislation': [],
    'secondary sources': []
}

for html_file in html_files:
    print(f"\nProcessing: {html_file.name}")
    
    with open(html_file, 'r', encoding='utf-8') as f:
        html_content = f.read()
    
    annotations = extract_parent_level_annotations(html_content)
    
    # Aggregate results
    for label_type in ['decision', 'legislation', 'secondary sources']:
        count = len(annotations[label_type])
        print(f"  - {label_type}: {count} annotations")
        all_annotations[label_type].extend(annotations[label_type])

print("\n" + "="*60)
print("TOTAL ANNOTATIONS ACROSS ALL FILES:")
for label_type in ['decision', 'legislation', 'secondary sources']:
    print(f"  {label_type}: {len(all_annotations[label_type])}")
print("="*60)


Processing: 1997CanLII16226_ONCA_annotated_EG_tech.html
  - decision: 248 annotations
  - legislation: 355 annotations
  - secondary sources: 15 annotations

Processing: 2021QCCA1675_annotated_EG_tech.html
  - decision: 53 annotations
  - legislation: 45 annotations
  - secondary sources: 1 annotations

Processing: 1989CanLII1415CITT_annotated_GL.html
  - decision: 17 annotations
  - legislation: 34 annotations
  - secondary sources: 33 annotations

Processing: 2001CanLII21117QCTDP_annotated_GL_tech.html
  - decision: 63 annotations
  - legislation: 245 annotations
  - secondary sources: 11 annotations

Processing: 2019SCC65_annotated_EG_revRL_tech.html
  - decision: 1003 annotations
  - legislation: 263 annotations
  - secondary sources: 179 annotations

Processing: 2024NBKB203_annotated_VP.html
  - decision: 143 annotations
  - legislation: 64 annotations
  - secondary sources: 24 annotations

TOTAL ANNOTATIONS ACROSS ALL FILES:
  decision: 1527
  legislation: 1006
  secondary sourc

In [6]:
print(f"DECISION ANNOTATIONS ({len(all_annotations['decision'])} total)\\n")
print("="*80)

for i, annotation in enumerate(all_annotations['decision'][:20], 1):  # Show first 20
    print(f"\n[{i}] DocID: {annotation['docid']}")
    print(f"    URI: {annotation['uri']}")
    print(f"    Sublabels: {', '.join(set(annotation['all_sublabels']))}")
    print(f"    Text: {annotation['text_content'][:100]}...")
    print(f"    HTML Preview: {annotation['full_html'][:200]}...")

if len(all_annotations['decision']) > 20:
    print(f"\n... and {len(all_annotations['decision']) - 20} more decision annotations")

DECISION ANNOTATIONS (1527 total)\n

[1] DocID: Church of Scientology ONCA 1997
    URI: https://canlii.ca/t/6hxv
    Sublabels: title
    Text: R. v. Church of Scientology...
    HTML Preview: <manual_label docid="Church of Scientology ONCA 1997" labelname="decision" parent="" style="background-color: rgb(106, 163, 255); color: black;" uri="https://canlii.ca/t/6hxv" verified="false"><manual...

[2] DocID: Church of Scientology ONCA 1997
    URI: https://canlii.ca/t/6hxv
    Sublabels: citation
    Text: 33 O.R. (3d) 65...
    HTML Preview: <manual_label docid="Church of Scientology ONCA 1997" labelname="decision" parent="" style="background-color: rgb(106, 163, 255); color: black;" uri="https://canlii.ca/t/6hxv" verified="false"><manual...

[3] DocID: Church of Scientology ONCA 1997
    URI: https://canlii.ca/t/6hxv
    Sublabels: citation
    Text: [1997] O.J. No. 1548...
    HTML Preview: <manual_label docid="Church of Scientology ONCA 1997" labelname="decision" parent="" style="bac

In [7]:
print(f"LEGISLATION ANNOTATIONS ({len(all_annotations['legislation'])} total)\\n")
print("="*80)

for i, annotation in enumerate(all_annotations['legislation'][:20], 1):  # Show first 20
    print(f"\n[{i}] DocID: {annotation['docid']}")
    print(f"    URI: {annotation['uri']}")
    print(f"    Sublabels: {', '.join(set(annotation['all_sublabels']))}")
    print(f"    Text: {annotation['text_content'][:100]}...")
    print(f"    HTML Preview: {annotation['full_html'][:200]}...")

if len(all_annotations['legislation']) > 20:
    print(f"\n... and {len(all_annotations['legislation']) - 20} more legislation annotations")

LEGISLATION ANNOTATIONS (1006 total)\n

[1] DocID: Charter
    URI: https://canlii.ca/t/ldsx
    Sublabels: title
    Text: Charter of Rights and Freedoms...
    HTML Preview: <manual_label docid="Charter" labelname="legislation" parent="" style="background-color: rgb(118, 206, 222); color: black;" uri="https://canlii.ca/t/ldsx" verified="false"><manual_label labelname="tit...

[2] DocID: Charter
    URI: https://canlii.ca/t/ldsx
    Sublabels: title, fragment
    Text: s. 8ofCharter...
    HTML Preview: <manual_label docid="Charter" labelname="legislation" parent="" style="background-color: rgb(118, 206, 222); color: black;" uri="https://canlii.ca/t/ldsx" verified="false"><manual_label fragmentid="se...

[3] DocID: Charter
    URI: https://canlii.ca/t/ldsx
    Sublabels: title, fragment
    Text: Canadian Charter of
Rights and Freedoms,ss. 8,24(2)...
    HTML Preview: <manual_label docid="Charter" labelname="legislation" parent="" style="background-color: rgb(118, 206, 222); color: bl

In [8]:
print(f"SECONDARY SOURCE ANNOTATIONS ({len(all_annotations['secondary sources'])} total)\\n")
print("="*80)

if len(all_annotations['secondary sources']) == 0:
    print("No secondary source annotations found in the analyzed files.")
else:
    for i, annotation in enumerate(all_annotations['secondary sources'][:20], 1):
        print(f"\n[{i}] DocID: {annotation['docid']}")
        print(f"    URI: {annotation['uri']}")
        print(f"    Sublabels: {', '.join(set(annotation['all_sublabels']))}")
        print(f"    Text: {annotation['text_content'][:100]}...")
        print(f"    HTML Preview: {annotation['full_html'][:200]}...")
    
    if len(all_annotations['secondary sources']) > 20:
        print(f"\n... and {len(all_annotations['secondary sources']) - 20} more secondary source annotations")

SECONDARY SOURCE ANNOTATIONS (263 total)\n

[1] DocID: Study of the Constitution
    URI: None
    Sublabels: authors, fragment, title, source
    Text: Dicey,Introduction to the Study of the Law of the Constitution,10th ed.
(1959),p. 193...
    HTML Preview: <manual_label docid="Study of the Constitution" labelname="secondary sources" parent="" style="background-color: rgb(39, 143, 227); color: white;" uri="None" verified="false"><manual_label labelname="...

[2] DocID: The Stranger in our Midst
    URI: None
    Sublabels: authors, title, source
    Text: Head, I., "The Stranger in our Midst: A Sketch of the Legal
Status of the Alien in Canada"(1964), Ca...
    HTML Preview: <manual_label docid="The Stranger in our Midst" labelname="secondary sources" parent="" style="background-color: rgb(39, 143, 227); color: white;" uri="None" verified="false"> <manual_label labelname=...

[3] DocID: Reform of the Criminal Jury
    URI: https://canlii.ca/t/2blz
    Sublabels: authors, fragment, ti

In [9]:
from collections import Counter

def analyze_sublabel_patterns(annotations, label_type):
    """Analyze which sublabels appear and in what patterns."""
    
    sublabel_counts = Counter()
    pattern_counts = Counter()
    
    for ann in annotations:
        # Count individual sublabels
        for sublabel in ann['all_sublabels']:
            sublabel_counts[sublabel] += 1
        
        # Count patterns (combinations of sublabels)
        pattern = tuple(sorted(set(ann['all_sublabels'])))
        pattern_counts[pattern] += 1
    
    print(f"\n{label_type.upper()} - SUBLABEL ANALYSIS")
    print("="*60)
    
    print("\nMost common sublabels:")
    for sublabel, count in sublabel_counts.most_common(10):
        print(f"  {sublabel}: {count}")
    
    print("\nMost common sublabel patterns:")
    for pattern, count in pattern_counts.most_common(10):
        print(f"  {pattern}: {count}")
    
    return sublabel_counts, pattern_counts

# Analyze each type
for label_type in ['decision', 'legislation', 'secondary sources']:
    if all_annotations[label_type]:
        analyze_sublabel_patterns(all_annotations[label_type], label_type)


DECISION - SUBLABEL ANALYSIS

Most common sublabels:
  title: 1309
  citation: 1236
  fragment: 856
  source: 2

Most common sublabel patterns:
  ('citation', 'title'): 416
  ('title',): 319
  ('fragment', 'title'): 312
  ('citation', 'fragment', 'title'): 254
  ('fragment',): 191
  ('citation',): 23
  (): 8
  ('citation', 'fragment'): 3
  ('fragment', 'source', 'title'): 1

LEGISLATION - SUBLABEL ANALYSIS

Most common sublabels:
  fragment: 842
  title: 571
  citation: 97

Most common sublabel patterns:
  ('fragment',): 424
  ('fragment', 'title'): 269
  ('title',): 218
  ('citation', 'fragment', 'title'): 39
  ('citation', 'title'): 37
  ('citation',): 17
  ('citation', 'fragment'): 1
  (): 1

SECONDARY SOURCES - SUBLABEL ANALYSIS

Most common sublabels:
  authors: 219
  title: 204
  source: 162
  fragment: 130
  citation: 1

Most common sublabel patterns:
  ('authors', 'source', 'title'): 77
  ('authors', 'fragment', 'source', 'title'): 69
  ('authors', 'fragment'): 29
  ('authors'

In [10]:
# Create clean lists for each category
decision_list = all_annotations['decision']
legislation_list = all_annotations['legislation']
secondary_source_list = all_annotations['secondary sources']

print(f"Created lists:")
print(f"  - decision_list: {len(decision_list)} items")
print(f"  - legislation_list: {len(legislation_list)} items")
print(f"  - secondary_source_list: {len(secondary_source_list)} items")
print("\nThese lists are now available for further analysis in subsequent cells.")

Created lists:
  - decision_list: 1527 items
  - legislation_list: 1006 items
  - secondary_source_list: 263 items

These lists are now available for further analysis in subsequent cells.


In [11]:
import pandas as pd

def calculate_component_coverage(decision_list, legislation_list, secondary_source_list):
    """
    Calculate the percentage of annotations containing each component type.
    Returns a DataFrame for easy visualization.
    """
    
    # Define components to track
    components = ['Title', 'Citation', 'Fragment', 'Source', 'Authors']
    component_keys = ['title', 'citation', 'fragment', 'source', 'authors']
    
    # Count occurrences for each list
    def count_components(annotation_list, components_to_check):
        counts = {comp: 0 for comp in components_to_check}
        for annotation in annotation_list:
            sublabels = annotation['all_sublabels']
            for comp_key in components_to_check:
                if comp_key in sublabels:
                    counts[comp_key] += 1
        return counts
    
    # Get counts for each category
    decision_counts = count_components(decision_list, component_keys)
    legislation_counts = count_components(legislation_list, component_keys)
    secondary_counts = count_components(secondary_source_list, component_keys)
    
    # Calculate percentages
    decision_pcts = {k: (v / len(decision_list) * 100) if len(decision_list) > 0 else 0 
                     for k, v in decision_counts.items()}
    legislation_pcts = {k: (v / len(legislation_list) * 100) if len(legislation_list) > 0 else 0 
                        for k, v in legislation_counts.items()}
    secondary_pcts = {k: (v / len(secondary_source_list) * 100) if len(secondary_source_list) > 0 else 0 
                      for k, v in secondary_counts.items()}
    
    # Create a results dictionary
    results = {
        'Component': components,
        'Decisions': [decision_pcts[k] for k in component_keys],
        'Legislation': [legislation_pcts[k] for k in component_keys],
        'Secondary Sources': [secondary_pcts[k] for k in component_keys]
    }
    
    df = pd.DataFrame(results)
    
    # Print summary information
    print("COMPONENT COVERAGE ANALYSIS")
    print("="*80)
    print(f"\nDataset sizes:")
    print(f"  - Decisions:           n = {len(decision_list)}")
    print(f"  - Legislation:         n = {len(legislation_list)}")
    print(f"  - Secondary Sources:   n = {len(secondary_source_list)}")
    
    print(f"\nComponent Coverage Percentages:")
    print("-"*80)
    
    # Format and display the table
    print(f"{'Component':<20} {'Decisions':<15} {'Legislation':<15} {'Sec. Sources':<15}")
    print(f"{'':20} {'(n='+str(len(decision_list))+')':<15} {'(n='+str(len(legislation_list))+')':<15} {'(n='+str(len(secondary_source_list))+')':<15}")
    print("-"*80)
    
    for idx, row in df.iterrows():
        comp = row['Component']
        dec = row['Decisions']
        leg = row['Legislation']
        sec = row['Secondary Sources']
        
        # Format as percentages with 1 decimal place
        dec_str = f"{dec:.1f}%" if dec > 0 else "---"
        leg_str = f"{leg:.1f}%" if leg > 0 else "---"
        sec_str = f"{sec:.1f}%" if sec > 0 else "---"
        
        print(f"{comp:<20} {dec_str:<15} {leg_str:<15} {sec_str:<15}")
    
    print("="*80)
    
    return df

# Calculate and display the coverage table
coverage_df = calculate_component_coverage(decision_list, legislation_list, secondary_source_list)

COMPONENT COVERAGE ANALYSIS

Dataset sizes:
  - Decisions:           n = 1527
  - Legislation:         n = 1006
  - Secondary Sources:   n = 263

Component Coverage Percentages:
--------------------------------------------------------------------------------
Component            Decisions       Legislation     Sec. Sources   
                     (n=1527)        (n=1006)        (n=263)        
--------------------------------------------------------------------------------
Title                85.3%           56.0%           76.4%          
Citation             45.6%           9.3%            0.4%           
Fragment             49.8%           72.9%           47.1%          
Source               0.1%            ---             61.6%          
Authors              ---             ---             83.3%          


In [13]:
def simplify_html(html_str):
    """
    Simplify HTML by converting <manual_label labelname="X"> to <X>.
    """
    import re
    
    # Use regex to replace manual_label tags with simplified versions
    def replace_manual_label(match):
        # Extract labelname attribute value
        labelname_match = re.search(r'labelname=["\']([^"\']+)["\']', match.group(0))
        if labelname_match:
            labelname = labelname_match.group(1)
            return f'<{labelname}>'
        return match.group(0)
    
    # Replace opening manual_label tags
    result = re.sub(r'<manual_label[^>]*>', replace_manual_label, html_str)
    # Replace closing manual_label tags
    result = re.sub(r'</manual_label>', '', result)
    
    return result


def analyze_sublabel_order(annotations, label_type, examples=0):
    """
    Analyze the order in which sublabels appear within each annotation.
    Returns ordered patterns showing which sublabels come before others.
    
    Args:
        annotations: List of annotation dictionaries
        label_type: Type of label ('decision', 'legislation', etc.)
        examples: Number of examples to show per pattern (default: 0 = no examples)
    """
    from collections import Counter, defaultdict
    import re
    
    ordered_pattern_counts = Counter()
    pattern_examples = defaultdict(list)  # Store examples for each pattern
    
    for ann in annotations:
        sublabels_in_order = []
        html_content = ann['full_html']
        
        # Check if this is manual annotation format or JSON output format
        if 'manual_label' in html_content:
            # Manual annotation format: <manual_label labelname="X">
            # Find all manual_label tags in order
            pattern = r'<manual_label[^>]*labelname=["\']([^"\']+)["\'][^>]*>'
            matches = re.finditer(pattern, html_content)
            for match in matches:
                sublabel_name = match.group(1)
                sublabels_in_order.append(sublabel_name)
        else:
            # JSON output format: simple tags like <title>, <fragment>, etc.
            # Find all sublabel tags in order
            sublabel_names = ['title', 'fragment', 'citation', 'source', 'authors']
            
            # Find all tag positions and sort by position
            tag_positions = []
            for sublabel_name in sublabel_names:
                pattern = f'<{sublabel_name}>'
                for match in re.finditer(pattern, html_content, re.IGNORECASE):
                    tag_positions.append((match.start(), sublabel_name))
            
            # Sort by position to maintain order
            tag_positions.sort(key=lambda x: x[0])
            sublabels_in_order = [name for pos, name in tag_positions]
        
        # Create ordered pattern (as tuple to make it hashable)
        if sublabels_in_order:
            ordered_pattern = tuple(sublabels_in_order)
            ordered_pattern_counts[ordered_pattern] += 1
            
            # Store example if we need examples and haven't collected enough yet
            if examples > 0 and len(pattern_examples[ordered_pattern]) < examples:
                pattern_examples[ordered_pattern].append(ann)
    
    print(f"\n{label_type.upper()} - SUBLABEL ORDER ANALYSIS")
    print("="*60)
    print(f"Total annotations: {len(annotations)}")
    print(f"Unique ordered patterns: {len(ordered_pattern_counts)}")
    
    print("\nMost common ordered sublabel patterns:")
    for pattern, count in ordered_pattern_counts.most_common(20):
        # Format with arrows to show order
        pattern_str = " → ".join(pattern)
        print(f"  [{count:3d}x] {pattern_str}")
        
        # Show examples if requested
        if examples > 0 and pattern in pattern_examples:
            num_to_show = min(examples, len(pattern_examples[pattern]))
            print(f"        Examples ({num_to_show}):")
            for i, example in enumerate(pattern_examples[pattern][:num_to_show], 1):
                simplified = simplify_html(example['full_html'])
                # Truncate if too long
                if len(simplified) > 300:
                    simplified = simplified[:300] + "..."
                print(f"        {i}. {simplified}")
            print()
    
    return ordered_pattern_counts

# Analyze order for each type
print("\n" + "="*80)
print("ANALYZING SUBLABEL ORDER (which comes before which)")
print("="*80)

for label_type in ['decision', 'legislation', 'secondary sources']:
    if all_annotations[label_type]:
        ordered_patterns = analyze_sublabel_order(all_annotations[label_type], label_type, examples=0)
        print()  # Extra line between sections


ANALYZING SUBLABEL ORDER (which comes before which)

DECISION - SUBLABEL ORDER ANALYSIS
Total annotations: 1527
Unique ordered patterns: 39

Most common ordered sublabel patterns:
  [319x] decision → title
  [284x] decision → title → fragment
  [211x] decision → title → citation → citation
  [170x] decision → fragment
  [160x] decision → title → citation
  [138x] decision → title → citation → fragment
  [ 82x] decision → title → citation → citation → fragment
  [ 22x] decision → title → fragment → fragment
  [ 18x] decision → citation
  [ 17x] decision → fragment → fragment
  [ 11x] decision → title → citation → fragment → citation → fragment
  [ 10x] decision → title → citation → citation → fragment → fragment
  [  9x] decision → title → citation → citation → citation → citation → citation → citation
  [  8x] decision
  [  7x] decision → title → citation → citation → citation → citation
  [  7x] decision → title → citation → citation → citation → citation → citation → citation → cita

decision_list: 1527 items
1278 start with title : 319 only title
653 have the title + citation (should be enought for disambiguation)
the fragment is present in 734 (170 time it is alone (fragment only = hard to disambiguate))
so decision usually start with a title 9when there is a title it is at the begining (before fragment and citation)
legislation_list: 1006 items
legilstation contain very often a fragment (we don't cite a legisltation without a specifique  fragment) -> 725 with 687 times it starts with the fragment. and 424 the fragment is alone (really hard to disambiguate)
otherwise it starts with the title (290). The citation is generaly accompagnated by the title (easy for disambiguate) and represnte  89 cases.

secondary_source_list: 263 items
for the sec source. The most comun pattern is authors, title source (fragment eventually) 54%
The fragment is presetn in 47%.1 cases (10 times alone (hard to disambiguate))
The source is there 9supposed easy to disambiguate in 162 cases

level 1 disambiguiation all is presetn (authors, title, source) (fragment or not) 55.5 (source is presente : 61.6 %)
level 2 : withotu source, but title (eventually + authors + fragment) : 14.1
level 3 : authors : 19.4
level 4 : fragment only : 3.8%

# Analysis of Few-Shot Examples from JSON

Now analyzing the selected few-shot examples from `combined_v3_with_sources_fixed_spacing.json`

In [17]:
import json
from pathlib import Path
from collections import Counter, defaultdict
from bs4 import BeautifulSoup

# Load the JSON file
json_file = Path.cwd().parent / 'few_shot_selection_tool' / 'second_selected' / 'examples_selected_45.json'

print(f"Loading: {json_file.name}")
print(f"File exists: {json_file.exists()}")

with open(json_file, 'r', encoding='utf-8') as f:
    few_shot_data = json.load(f)

print(f"\nTotal examples in file: {len(few_shot_data)}")

# Filter for selected examples only
selected_examples = [item for item in few_shot_data if item.get('selected', False) == True]

print(f"Selected examples: {len(selected_examples)}")
print(f"\nFirst selected example preview:")
if selected_examples:
    print(f"  Source file: {selected_examples[0].get('source_file', 'N/A')}")
    print(f"  Output preview: {selected_examples[0]['example']['output'][:150]}...")

Loading: examples_selected_45.json
File exists: True

Total examples in file: 787
Selected examples: 45

First selected example preview:
  Source file: N/A
  Output preview:  Peter
C. Engelmann, for the respondent Statutes Cited: <legislation> <title>Canadian
International Trade Tribunal Act</title>, <citation>S.C. 1988, c...


In [18]:
def extract_annotations_from_json_output(output_text):
    """
    Extract all parent-level annotations from the JSON output field using regex.
    The output contains XML-like tags: <legislation>, <decision>, <secondary sources>
    with nested tags like <title>, <fragment>, <citation>, <source>, <authors>
    
    Returns a dict with keys: 'decision', 'legislation', 'secondary sources'
    Each value is a list of annotation dictionaries.
    """
    import re
    
    annotations = {
        'decision': [],
        'legislation': [],
        'secondary sources': []
    }
    
    # Define patterns for each label type
    # Note: 'secondary sources' has a space, so we use \s+ to match it
    label_patterns = {
        'decision': (r'<decision>', r'</decision>'),
        'legislation': (r'<legislation>', r'</legislation>'),
        'secondary sources': (r'<secondary\s+sources>', r'</secondary\s+sources>')
    }
    
    for label_type, (open_tag, close_tag) in label_patterns.items():
        # Find all occurrences of this tag with balanced opening/closing
        pos = 0
        while pos < len(output_text):
            # Find next opening tag
            open_match = re.search(open_tag, output_text[pos:], re.IGNORECASE)
            if not open_match:
                break
            
            start_pos = pos + open_match.start()
            content_start = pos + open_match.end()
            
            # Find matching closing tag by counting depth
            depth = 1
            search_pos = content_start
            
            while depth > 0 and search_pos < len(output_text):
                # Find next opening or closing tag
                next_open = re.search(open_tag, output_text[search_pos:], re.IGNORECASE)
                next_close = re.search(close_tag, output_text[search_pos:], re.IGNORECASE)
                
                open_pos = search_pos + next_open.start() if next_open else len(output_text)
                close_pos = search_pos + next_close.start() if next_close else len(output_text)
                
                if close_pos < open_pos:
                    # Found a closing tag
                    depth -= 1
                    if depth == 0:
                        # This is our matching closing tag
                        close_match_len = len(next_close.group(0))
                        end_pos = close_pos + close_match_len
                        
                        # Extract the full HTML and content
                        full_html = output_text[start_pos:end_pos]
                        content = output_text[content_start:close_pos]
                        
                        # Extract all sublabels using regex
                        all_sublabels = []
                        sublabel_names = ['title', 'fragment', 'citation', 'source', 'authors']
                        
                        for sublabel_name in sublabel_names:
                            # Count occurrences of opening sublabel tags
                            sublabel_pattern = f'<{sublabel_name}>'
                            matches = re.findall(sublabel_pattern, content, re.IGNORECASE)
                            all_sublabels.extend([sublabel_name] * len(matches))
                        
                        # Extract text content (remove all tags)
                        text_content = re.sub(r'<[^>]+>', '', content).strip()
                        # Clean up extra whitespace
                        text_content = re.sub(r'\s+', ' ', text_content)
                        
                        annotation_data = {
                            'full_html': full_html,
                            'text_content': text_content,
                            'direct_sublabels': [],  # Not computed separately
                            'all_sublabels': all_sublabels
                        }
                        
                        annotations[label_type].append(annotation_data)
                        
                        # Move position past this match
                        pos = end_pos
                        break
                    else:
                        search_pos = close_pos + len(next_close.group(0))
                elif open_pos < len(output_text):
                    # Found an opening tag (nested)
                    depth += 1
                    search_pos = open_pos + len(next_open.group(0))
                else:
                    # No more tags found
                    break
            else:
                # If we didn't find a matching closing tag, move past this opening tag
                pos = content_start
    
    return annotations

# Test the extraction on first selected example
if selected_examples:
    test_annotations = extract_annotations_from_json_output(selected_examples[0]['example']['output'])
    print("Test extraction on first example:")
    for label_type, anns in test_annotations.items():
        print(f"  {label_type}: {len(anns)} annotations")

Test extraction on first example:
  decision: 0 annotations
  legislation: 2 annotations
  secondary sources: 0 annotations


In [19]:
# Process all selected examples
fewshot_annotations = {
    'decision': [],
    'legislation': [],
    'secondary sources': []
}

print("Processing selected few-shot examples...")
print("="*80)

for i, item in enumerate(selected_examples, 1):
    output_text = item['example']['output']
    annotations = extract_annotations_from_json_output(output_text)
    
    # Aggregate results
    for label_type in ['decision', 'legislation', 'secondary sources']:
        fewshot_annotations[label_type].extend(annotations[label_type])

print("\nTOTAL ANNOTATIONS FROM FEW-SHOT EXAMPLES:")
print("="*80)
for label_type in ['decision', 'legislation', 'secondary sources']:
    print(f"  {label_type}: {len(fewshot_annotations[label_type])} annotations")
print("="*80)

Processing selected few-shot examples...

TOTAL ANNOTATIONS FROM FEW-SHOT EXAMPLES:
  decision: 61 annotations
  legislation: 20 annotations
  secondary sources: 35 annotations


## Preview of Few-Shot Annotations

In [20]:
print(f"DECISION ANNOTATIONS FROM FEW-SHOT ({len(fewshot_annotations['decision'])} total)\n")
print("="*80)

for i, annotation in enumerate(fewshot_annotations['decision'][:15], 1):
    print(f"\n[{i}]")
    print(f"    Sublabels: {', '.join(set(annotation['all_sublabels']))}")
    print(f"    Text: {annotation['text_content'][:80]}...")
    print(f"    XML: {annotation['full_html'][:120]}...")

if len(fewshot_annotations['decision']) > 15:
    print(f"\n... and {len(fewshot_annotations['decision']) - 15} more decision annotations")

DECISION ANNOTATIONS FROM FEW-SHOT (61 total)


[1]
    Sublabels: title
    Text: Pfizer case (supra)...
    XML: <decision><title>Pfizer case</title> (supra)</decision>...

[2]
    Sublabels: title
    Text: Olympia Floor and Wall Tile Company v. The Deputy Minister of National Revenue f...
    XML: <decision><title>Olympia Floor and Wall Tile Company v. The Deputy Minister of
National Revenue for Customs and Excise</...

[3]
    Sublabels: citation
    Text: [1977] 1 S.C.R. 456...
    XML: <decision><citation>[1977] 1 S.C.R. 456</citation></decision>...

[4]
    Sublabels: citation
    Text: 2 T.B.R. 106...
    XML: <decision><citation>2 T.B.R. 106</citation></decision>...

[5]
    Sublabels: citation
    Text: 5 C.E.R. 562...
    XML: <decision><citation>5 C.E.R. 562</citation></decision>...

[6]
    Sublabels: fragment
    Text: Ibid, at p. 460...
    XML: <decision>Ibid, at <fragment>p. 460</fragment></decision>...

[7]
    Sublabels: citation
    Text: [1970] Ex. C.R. 828...
   

In [21]:
print(f"LEGISLATION ANNOTATIONS FROM FEW-SHOT ({len(fewshot_annotations['legislation'])} total)\n")
print("="*80)

for i, annotation in enumerate(fewshot_annotations['legislation'][:15], 1):
    print(f"\n[{i}]")
    print(f"    Sublabels: {', '.join(set(annotation['all_sublabels']))}")
    print(f"    Text: {annotation['text_content'][:80]}...")
    print(f"    XML: {annotation['full_html'][:120]}...")

if len(fewshot_annotations['legislation']) > 15:
    print(f"\n... and {len(fewshot_annotations['legislation']) - 15} more legislation annotations")

LEGISLATION ANNOTATIONS FROM FEW-SHOT (20 total)


[1]
    Sublabels: title, fragment, citation
    Text: Canadian International Trade Tribunal Act, S.C. 1988, c. 56,subs.54(2) and s. 60...
    XML: <legislation> <title>Canadian
International Trade Tribunal Act</title>, <citation>S.C. 1988, c. 56</citation>,<fragment>...

[2]
    Sublabels: title, fragment, citation
    Text: Excise Tax Act, R.S.C. 1970, c. E-13, paragraph 1(h), Part XII, Schedule III...
    XML: <legislation><title>Excise
Tax Act</title>, <citation>R.S.C. 1970, c. E-13</citation>, <fragment>paragraph 1(h), Part XI...

[3]
    Sublabels: fragment
    Text: exemption clause...
    XML: <legislation><fragment>exemption clause</fragment></legislation>...

[4]
    Sublabels: fragment
    Text: exemption clause...
    XML: <legislation><fragment>exemption clause</fragment></legislation>...

[5]
    Sublabels: citation
    Text: S.C. 1988, c. 56...
    XML: <legislation><citation>S.C. 1988, c. 56</citation></legislation>...


In [22]:
print(f"SECONDARY SOURCE ANNOTATIONS FROM FEW-SHOT ({len(fewshot_annotations['secondary sources'])} total)\n")
print("="*80)

if len(fewshot_annotations['secondary sources']) == 0:
    print("No secondary source annotations found in the selected few-shot examples.")
else:
    for i, annotation in enumerate(fewshot_annotations['secondary sources'][:15], 1):
        print(f"\n[{i}]")
        print(f"    Sublabels: {', '.join(set(annotation['all_sublabels']))}")
        print(f"    Text: {annotation['text_content'][:80]}...")
        print(f"    XML: {annotation['full_html'][:120]}...")
    
    if len(fewshot_annotations['secondary sources']) > 15:
        print(f"\n... and {len(fewshot_annotations['secondary sources']) - 15} more secondary source annotations")

SECONDARY SOURCE ANNOTATIONS FROM FEW-SHOT (35 total)


[1]
    Sublabels: title
    Text: Funk &amp; Wagnalls New Standard Dictionary of the English Language...
    XML: <secondary sources><title>Funk
&amp; Wagnalls New Standard Dictionary of the English Language</title></secondary sources...

[2]
    Sublabels: title, source
    Text: The Oxford English Dictionary (Second Edition)...
    XML: <secondary sources><title>The Oxford
English Dictionary</title> (<source>Second Edition</source>)</secondary sources>...

[3]
    Sublabels: title, source
    Text: Black's Law Dictionary (Revised Fourth Edition)...
    XML: <secondary sources><title>Black's Law Dictionary </title>(<source>Revised
Fourth Edition</source>)</secondary sources>...

[4]
    Sublabels: title
    Text: The Houghton Mifflin Canadian Dictionary of the English Language...
    XML: <secondary sources><title>The Houghton Mifflin Canadian Dictionary of the English
Language</title></secondary sources>...

[5]
    Sublabels: 

## Sublabel Pattern Analysis for Few-Shot Examples

In [35]:
# Analyze sublabel patterns for few-shot examples
print("\n" + "="*80)
print("SUBLABEL PATTERN ANALYSIS - FEW-SHOT EXAMPLES")
print("="*80)

for label_type in ['decision', 'legislation', 'secondary sources']:
    if fewshot_annotations[label_type]:
        analyze_sublabel_patterns(fewshot_annotations[label_type], label_type)


SUBLABEL PATTERN ANALYSIS - FEW-SHOT EXAMPLES

DECISION - SUBLABEL ANALYSIS

Most common sublabels:
  fragment: 76
  citation: 74
  title: 56

Most common sublabel patterns:
  ('citation', 'fragment', 'title'): 33
  ('fragment',): 19
  ('citation', 'title'): 10
  ('fragment', 'title'): 7
  ('title',): 6
  ('citation',): 5
  (): 3
  ('citation', 'fragment'): 2

LEGISLATION - SUBLABEL ANALYSIS

Most common sublabels:
  fragment: 26
  title: 22
  citation: 7

Most common sublabel patterns:
  ('fragment', 'title'): 11
  ('title',): 8
  ('fragment',): 8
  ('citation', 'fragment', 'title'): 3
  ('citation',): 3

SECONDARY SOURCES - SUBLABEL ANALYSIS

Most common sublabels:
  title: 24
  source: 17
  authors: 17
  fragment: 10

Most common sublabel patterns:
  ('authors', 'fragment', 'source', 'title'): 8
  ('title',): 6
  ('source', 'title'): 4
  ('authors', 'source', 'title'): 4
  ('authors',): 4
  ('authors', 'title'): 1
  ('fragment',): 1
  ('fragment', 'source', 'title'): 1


## Sublabel Order Analysis for Few-Shot Examples

In [ ]:
# Analyze order for few-shot examples
print("\n" + "="*80)
print("ANALYZING SUBLABEL ORDER (which comes before which) - FEW-SHOT EXAMPLES")
print("="*80)

for label_type in ['decision', 'legislation', 'secondary sources']:
    if fewshot_annotations[label_type]:
        ordered_patterns = analyze_sublabel_order(fewshot_annotations[label_type], label_type, examples=3)
        print()  # Extra line between sections

## Comparison: Manual Annotations vs Few-Shot Examples

In [23]:
def compare_annotation_patterns(manual_annotations, fewshot_annotations):
    """
    Compare pattern distributions between manual annotations and few-shot examples.
    """
    print("\n" + "="*80)
    print("COMPARISON: MANUAL ANNOTATIONS vs FEW-SHOT EXAMPLES")
    print("="*80)
    
    for label_type in ['decision', 'legislation', 'secondary sources']:
        print(f"\n{'='*80}")
        print(f"{label_type.upper()}")
        print(f"{'='*80}")
        
        manual = manual_annotations[label_type]
        fewshot = fewshot_annotations[label_type]
        
        print(f"\nTotal counts:")
        print(f"  Manual annotations:  {len(manual):3d}")
        print(f"  Few-shot examples:   {len(fewshot):3d}")
        
        if not manual and not fewshot:
            print(f"  No annotations of type '{label_type}' in either dataset.")
            continue
        
        # Compare sublabel frequency
        manual_sublabels = Counter()
        fewshot_sublabels = Counter()
        
        for ann in manual:
            for sublabel in ann['all_sublabels']:
                manual_sublabels[sublabel] += 1
        
        for ann in fewshot:
            for sublabel in ann['all_sublabels']:
                fewshot_sublabels[sublabel] += 1
        
        print(f"\nSublabel usage comparison:")
        print(f"  {'Sublabel':<20} {'Manual':>10} {'Few-shot':>10} {'Difference':>12}")
        print(f"  {'-'*20} {'-'*10} {'-'*10} {'-'*12}")
        
        all_sublabels = set(manual_sublabels.keys()) | set(fewshot_sublabels.keys())
        for sublabel in sorted(all_sublabels):
            manual_count = manual_sublabels.get(sublabel, 0)
            fewshot_count = fewshot_sublabels.get(sublabel, 0)
            diff = fewshot_count - manual_count
            diff_str = f"{diff:+d}" if diff != 0 else "0"
            print(f"  {sublabel:<20} {manual_count:>10} {fewshot_count:>10} {diff_str:>12}")
        
        # Compare pattern diversity
        manual_patterns = Counter()
        fewshot_patterns = Counter()
        
        for ann in manual:
            pattern = tuple(sorted(set(ann['all_sublabels'])))
            manual_patterns[pattern] += 1
        
        for ann in fewshot:
            pattern = tuple(sorted(set(ann['all_sublabels'])))
            fewshot_patterns[pattern] += 1
        
        print(f"\nPattern diversity:")
        print(f"  Manual annotations:  {len(manual_patterns)} unique patterns")
        print(f"  Few-shot examples:   {len(fewshot_patterns)} unique patterns")
        
        # Show common patterns
        print(f"\nTop 5 patterns in manual annotations:")
        for i, (pattern, count) in enumerate(manual_patterns.most_common(5), 1):
            pattern_str = ", ".join(pattern) if pattern else "(no sublabels)"
            print(f"    {i}. [{count:2d}x] {pattern_str}")
        
        print(f"\nTop 5 patterns in few-shot examples:")
        for i, (pattern, count) in enumerate(fewshot_patterns.most_common(5), 1):
            pattern_str = ", ".join(pattern) if pattern else "(no sublabels)"
            print(f"    {i}. [{count:2d}x] {pattern_str}")

# Run the comparison
compare_annotation_patterns(all_annotations, fewshot_annotations)


COMPARISON: MANUAL ANNOTATIONS vs FEW-SHOT EXAMPLES

DECISION

Total counts:
  Manual annotations:  1527
  Few-shot examples:    61

Sublabel usage comparison:
  Sublabel                 Manual   Few-shot   Difference
  -------------------- ---------- ---------- ------------
  citation                   1236         59        -1177
  fragment                    856         56         -800
  source                        2          0           -2
  title                      1309         46        -1263

Pattern diversity:
  Manual annotations:  9 unique patterns
  Few-shot examples:   6 unique patterns

Top 5 patterns in manual annotations:
    1. [416x] citation, title
    2. [319x] title
    3. [312x] fragment, title
    4. [254x] citation, fragment, title
    5. [191x] fragment

Top 5 patterns in few-shot examples:
    1. [26x] citation, fragment, title
    2. [11x] fragment, title
    3. [ 9x] fragment
    4. [ 7x] citation
    5. [ 4x] title

LEGISLATION

Total counts:
  Manual a